# Object Detection and Person Layout on PASCAL VOC 2012

**One main notebook trains both tasks, evaluates them and exports all four deployment files.**

- **Object detection:** Faster R-CNN + MobileNetV3-Large FPN detects 20 whole-object classes, including people.
- **Person layout:** a second Faster R-CNN + MobileNetV3-Large FPN learns head/hand/foot boxes for each specified person. The part head needs new training; the original object checkpoint cannot detect these part classes.

Run all code cells from top to bottom. Sections **1–10** cover objects; **11–19** cover body parts; section **20** combines the results and deployment files. Both tasks include cleaning, preprocessing, fine-tuning, hyperparameter tuning, evaluation and optimization. Implementation functions remain in `src/` for readability; the project ZIP contains them.

The full-image demo first detects people, then predicts parts in each person's context crop. Layout evaluation supplies known person regions, so its metrics do not measure first-stage person-detection errors. No actual VOC training results or trained weights are bundled.

Start with **QUICK_RUN=True** to check execution; use **False** for measured full-data experiments. Free GPU runtime limits vary, and full mode runs four object trials plus four layout trials. Save checkpoints/results externally before the runtime ends.


## 0. Setup — upload the complete project once

On Colab, enable a GPU and run the next cell. Select **voc_detection_layout_project.zip** if prompted; it extracts all `.py` files. On your computer, extract that ZIP and start Jupyter from its `voc_detection_layout` folder. No separate layout notebook is needed.

Use a fresh runtime after switching from an older project version so Python does not reuse cached old modules. Install missing dependencies with the commented pip cell and restart if required. A compatible CUDA torch/torchvision pair is normally preinstalled in Colab; do not replace it with CPU wheels.


In [ ]:
from pathlib import Path
import os
if not Path('src/layout_data.py').exists():
    if Path('/content/voc_detection_layout/src/layout_data.py').exists():
        os.chdir('/content/voc_detection_layout')
    else:
        from google.colab import files
        import zipfile
        uploaded = files.upload()  # Select voc_detection_layout_project.zip
        archives = [name for name in uploaded if name.endswith('.zip')]
        if len(archives) != 1:
            raise ValueError('Select exactly one project ZIP.')
        with zipfile.ZipFile(archives[0]) as archive:
            archive.extractall('/content')
        os.chdir('/content/voc_detection_layout')
assert Path('src/layout_data.py').exists(), 'Use the updated project ZIP.'
print('Project directory:', Path.cwd())
# Install missing dependencies if needed, then restart the runtime:
# %pip install -r requirements.txt

In [ ]:
# Uncomment only if dependencies are missing, then restart the kernel.
# %pip install -r requirements.txt
from pathlib import Path
assert Path('src').is_dir(), 'Change the working directory to the extracted voc_detection folder.'
import torch, torchvision
print('PyTorch:', torch.__version__, 'Torchvision:', torchvision.__version__)
print('CUDA available:', torch.cuda.is_available())

In [ ]:
import json
import shutil
import gc
from dataclasses import replace
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from IPython.display import display
from src.config import Config, CLASSES, LAYOUT_CLASSES, seed_everything, write_json
from src.data import download_voc, audit_voc, make_splits, class_counts, loader, VOCDataset
from src.model import build_model, save_checkpoint, load_checkpoint
from src.engine import train_experiment, predict_loader, benchmark
from src.metrics import evaluate_detections, error_analysis
from src.visualize import draw_detections
from src.layout_data import audit_layout, make_layout_splits

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# A fresh directory prevents accidental overwriting or mixing experiment results.
RUN_NAME = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%f')
RUN_DIR = (
    Path("/content/drive/MyDrive/voc_detection_runs")
    / RUN_NAME
)
RUN_DIR.mkdir(parents=True, exist_ok=False)
DATA_ROOT = Path('data')
QUICK_RUN = False  # First verify the workflow; set False for submission experiments.
CFG = Config(task='objects')
# Configure layout training here too; the same QUICK_RUN flag controls both tasks.
LAYOUT_CFG = Config(task='layout', epochs=12, lr=.005, batch_size=2,
                    min_size=480, max_size=800, crop_padding=.35)
if QUICK_RUN:
    CFG = replace(CFG, epochs=1, train_limit=96, tune_limit=48, eval_limit=48,
                  min_size=320, max_size=640)
if QUICK_RUN:
    LAYOUT_CFG = replace(LAYOUT_CFG, epochs=1, train_limit=32, tune_limit=16,
                         eval_limit=32, min_size=320, max_size=640)
seed_everything(CFG.seed)
write_json(RUN_DIR / 'environment.json', {
    'torch': str(torch.__version__), 'torchvision': str(torchvision.__version__),
    'device': str(DEVICE), 'quick_run': QUICK_RUN, 'config': CFG.dictionary()})
print('Device:', DEVICE, '| Run:', RUN_DIR, '| Quick run:', QUICK_RUN)

## 1. Data cleaning
Download the official VOC 2012 train/validation archive (several GB; reserve roughly 8–10 GB of disk space for archive and extracted files). Training uses only official `train`; official `val` is held out for final evaluation. VOC test annotations are not provided here, so this is **held-out validation performance**, not an official test-server result.

We decode images, check XML, verify class names and finite coordinates, clip boxes to image bounds, discard invalid boxes, and log every change. Images with no valid boxes remain background examples. We remove exact decoded-pixel duplicates, keeping official validation images first. Near-duplicates and possible overlap with pretrained COCO data are not audited.

VOC XML boxes use 1-based inclusive coordinates; the model uses 0-based half-open `XYXY`. We subtract one only from `xmin/ymin`. Training includes valid difficult objects; evaluation ignores difficult instances according to VOC matching rules.

In [ ]:
# If you already have the dataset, set VOC_DIR to .../VOCdevkit/VOC2012 instead.
VOC_DIR = download_voc(DATA_ROOT)
clean, cleaning_events = audit_voc(VOC_DIR, RUN_DIR)
print({name: len(records) for name, records in clean.items()})
print('Logged cleaning events:', len(cleaning_events))
display(pd.DataFrame(cleaning_events).head(15))

## 2. Preprocessing and exploratory analysis
Split official training images into **fit (80%)** and **tune (20%)**, using a fixed seed. Exact duplicates are removed before this split. Tune selects hyperparameters, checkpoints and deployment resolution; final evaluation is untouched until section 8.

The split is random, not multilabel-stratified. Check class coverage below, especially in quick mode. Classes without non-difficult ground truth have undefined AP and are excluded from that split's mean; compare models on identical records.

Preprocessing converts RGB pixels to float tensors in `[0,1]`. The detector internally resizes and normalizes inputs; do not normalize a second time. The only training augmentation is horizontal flip, applied to both image and boxes. Evaluation has no random augmentation.

In [ ]:
parts = make_splits(clean, CFG, RUN_DIR)
counts = pd.DataFrame({key: class_counts(records) for key, records in parts.items()})
display(counts)
print('Image counts:', {k: len(v) for k, v in parts.items()})
if (counts == 0).any().any():
    print('WARNING: Some classes lack ground truth. Use the full split for meaningful reporting.')
counts.plot.bar(figsize=(13, 4), title='Non-difficult object counts by class')
plt.tight_layout(); plt.show()
widths = [r['width'] for r in parts['fit']]
heights = [r['height'] for r in parts['fit']]
plt.scatter(widths, heights, s=8, alpha=.35)
plt.xlabel('Image width'); plt.ylabel('Image height'); plt.title('Training image sizes'); plt.show()

In [ ]:
# Ground truth visualization: confirm alignment before training.
sample = parts['fit'][0]
with Image.open(sample['path']) as im:
    truth = {'boxes': [o['box'] for o in sample['objects']],
             'labels': [o['label'] for o in sample['objects']],
             'scores': [1.0] * len(sample['objects'])}
    display(draw_detections(im, truth))
image_tensor, sample_target = VOCDataset([sample], flip_probability=1.0)[0]
flipped_image = Image.fromarray((image_tensor.permute(1,2,0).numpy()*255).astype('uint8'))
display(draw_detections(flipped_image, {
    **sample_target, 'scores': torch.ones(len(sample_target['boxes']))}))

## 3. Model and pretrained baseline
**Faster R-CNN + MobileNetV3-Large + Feature Pyramid Network:** the backbone extracts multiscale features; a region proposal network proposes boxes; ROI heads predict classes and refine box coordinates; NMS removes redundant boxes.

Use pretrained COCO weights to reduce data and compute needs. We copy the 20 corresponding classifier/regressor rows into a VOC-specific head, including COCO/VOC name aliases. This provides a meaningful zero-VOC-training baseline. Fine-tuning then adapts the detector to VOC.

**Limitation:** VOC classes overlap with COCO classes, and image overlap across pretraining corpora has not been audited. Baseline scores are not evidence of learning from scratch. The submission must disclose this initialization.

The backbone can be frozen for a cheaper adaptation experiment. Frozen batch normalization is kept consistent when saving/reloading checkpoints.

In [ ]:
seed_everything(CFG.seed)
model = build_model(CFG).to(DEVICE)
baseline_predictions, baseline_targets = predict_loader(model, loader(parts['tune'], CFG), DEVICE)
baseline_metrics = evaluate_detections(baseline_predictions, baseline_targets)
write_json(RUN_DIR / 'baseline_tune.json', baseline_metrics)
print('Pretrained baseline tune mAP50:', baseline_metrics['map50'])
del model
gc.collect()
if DEVICE.type == 'cuda': torch.cuda.empty_cache()

## 4. Fine-tuning
Optimize the sum of classifier, box-regression, RPN objectness and RPN box-regression losses with SGD. Use CUDA mixed precision when available, gradient clipping, a step learning-rate schedule and early stopping based on **tune mAP50**. Save the best checkpoint, full loss history and trial configuration.

The checkpoint is for inference, not an exact mid-epoch optimizer resume. If a free session disconnects, completed runs remain reusable if you saved them externally; an interrupted trial must restart. Save artifacts outside ephemeral notebook storage.

In [ ]:
baseline_cfg = replace(
    CFG, lr=.005, freeze_backbone=False, flip_probability=.5
)
frozen_result = train_experiment(baseline_cfg, parts['fit'], parts['tune'],
                                 RUN_DIR / 'frozen_backbone', DEVICE)

## 5. Hyperparameter tuning and controlled experiments
Use a small explicit search rather than an expensive blind sweep. The default trial changes both learning rate and backbone training relative to the frozen baseline, so **do not attribute that comparison to one factor**. The two full-backbone trials isolate learning rate. The no-flip trial isolates augmentation at the same learning rate.

Every trial uses the same seed, splits, resolution, epoch budget and pretrained initialization. Select by tune mAP50 only. Quick mode tests plumbing with tiny subsets and one epoch; it does not establish a reliable ranking. For stronger evidence, run full data and repeat the winning settings with several seeds.

In [ ]:
trial_configs = {
    'full_lr_005': replace(CFG, lr=.005, freeze_backbone=False),
    'full_lr_001': replace(CFG, lr=.001, freeze_backbone=False),
    'full_no_flip': replace(CFG, lr=.005, freeze_backbone=False, flip_probability=0.),
}
results = [frozen_result]
for name, trial_cfg in trial_configs.items():
    results.append(train_experiment(trial_cfg, parts['fit'], parts['tune'], RUN_DIR / name, DEVICE))
experiments = pd.DataFrame(results).sort_values('best_tune_map50', ascending=False)
experiments.to_csv(RUN_DIR / 'experiments.csv', index=False)
display(experiments[['run','lr','freeze_backbone','flip_probability',
                     'best_tune_map50','epochs_completed','seconds']])
best = experiments.iloc[0]
BEST_CHECKPOINT = Path(best['checkpoint'])
model, best_cfg, checkpoint_metadata = load_checkpoint(BEST_CHECKPOINT, DEVICE)
print('Selected checkpoint:', BEST_CHECKPOINT)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for result in results:
    history = pd.read_json(RUN_DIR / result['run'] / 'history.json')
    axes[0].plot(history['epoch'], history['train_loss'], marker='o', label=result['run'])
    axes[1].plot(history['epoch'], history['tune_map50'], marker='o', label=result['run'])
axes[0].set(title='Training loss', xlabel='Epoch', ylabel='Loss')
axes[1].set(title='Tuning mAP50', xlabel='Epoch', ylabel='mAP50', ylim=(0,1))
axes[1].legend(); plt.tight_layout()
plt.savefig(RUN_DIR / 'training_curves.png', dpi=150); plt.show()

## 6. Evaluation methodology and error analysis
**Primary metric:** VOC 2012-style continuous interpolated mAP at IoU > 0.50, averaged over classes with non-difficult ground truth. Predictions are sorted by confidence; each non-difficult object matches at most once; duplicate detections are false positives. Matches to difficult instances are ignored.

**Secondary metric:** `map50_95_voc` averages this VOC-style AP over IoU 0.50:0.05:0.95. It is **not official COCO mAP**, which uses different recall sampling and evaluation conventions. Max detections per image is 100; evaluation score floor is 0.001. The app's confidence slider does not change the AP calculation.

Analyze tuning errors now. Diagnostics at confidence 0.5 distinguish missed objects, duplicates, class mistakes, localization mistakes and background detections. These categories are a greedy heuristic, not a formal causal error decomposition.

In [ ]:
tune_predictions, tune_targets = predict_loader(model, loader(parts['tune'], best_cfg), DEVICE)
tune_metrics = evaluate_detections(tune_predictions, tune_targets)
write_json(RUN_DIR / 'selected_tune_metrics.json', tune_metrics)
errors = pd.DataFrame(error_analysis(tune_predictions, tune_targets))
errors['error_count'] = errors[['duplicate','wrong_class','localization','background','missed']].sum(axis=1)
errors.to_csv(RUN_DIR / 'tune_errors.csv', index=False)
display(errors.drop(columns=['image_index']).sum().to_frame('count'))
pd.Series(tune_metrics['per_class_ap50']).plot.bar(figsize=(12, 4), ylim=(0,1), title='Tuning AP50 by class')
plt.tight_layout(); plt.savefig(RUN_DIR / 'tune_ap50.png', dpi=150); plt.show()
for i in errors.nlargest(3, 'error_count')['image_index']:
    record = parts['tune'][int(i)]
    with Image.open(record['path']) as im:
        print(record['id'], errors.loc[errors['image_index'] == i].to_dict('records'))
        display(draw_detections(im, tune_predictions[int(i)]))

## 7. Optimization: inference resolution versus latency
Try a smaller input resolution on the **same selected checkpoint**. We measure quality and synchronized batch-one model latency on tuning images. Timing includes tensor transfer, internal resize, detector and NMS, but excludes file decoding, UI rendering and network overhead. Preload at most 10 representative images and report warm-up, median, p95, device and thread count.

Choose the fastest policy only if it stays within an absolute **0.02 mAP50** drop on tuning data. This is a predefined deployment constraint, not a guaranteed speedup. Resolution changes compute, not parameter count or checkpoint size. If no faster policy qualifies, keep the original. GPU peak allocation is reported when CUDA is used; CPU memory is not measured. Both policies use FP32 inference for a fair resolution comparison.

In [ ]:
policies = [
    {'name': 'original', 'min_size': best_cfg.min_size, 'max_size': best_cfg.max_size},
    {'name': 'smaller', 'min_size': max(160, best_cfg.min_size * 2 // 3),
     'max_size': max(320, best_cfg.max_size * 2 // 3)},
]
MAX_MAP50_DROP = .02
optimization_rows = []
for policy in policies:
    model.transform.min_size = (policy['min_size'],)
    model.transform.max_size = policy['max_size']
    pred, gt = predict_loader(model, loader(parts['tune'], best_cfg), DEVICE)
    quality = evaluate_detections(pred, gt)
    timing = benchmark(model, VOCDataset(parts['tune']), DEVICE,
                       warmup=3, repeats=10 if QUICK_RUN else 30)
    optimization_rows.append({**policy, 'tune_map50': quality['map50'],
                               'tune_map50_95_voc': quality['map50_95_voc'],
                               'checkpoint_mb': BEST_CHECKPOINT.stat().st_size / 2**20,
                               'parameters': sum(p.numel() for p in model.parameters()), **timing})
optimization = pd.DataFrame(optimization_rows)
reference_ap = optimization_rows[0]['tune_map50']
assert reference_ap is not None, 'Need evaluable classes in tuning data.'
eligible = optimization[optimization['tune_map50'] >= reference_ap - MAX_MAP50_DROP]
selected = eligible.sort_values('median_ms', kind='stable').iloc[0]
SELECTED_POLICY = next(p for p in policies if p['name'] == selected['name'])
optimization.to_csv(RUN_DIR / 'optimization_tune.csv', index=False)
write_json(RUN_DIR / 'selected_policy.json', SELECTED_POLICY)
display(optimization)
print('Selected policy:', SELECTED_POLICY)

## 8. Final held-out evaluation
The model and deployment policy are now fixed. Evaluate original and selected deployment resolutions on official validation images (or the clearly labeled quick subset). Report both as a final comparison; do not change the chosen policy based on these scores. Repeated development against this set would make it a tuning set.

A full run excludes images rejected by cleaning; compare to external VOC results cautiously. `splits.json` records exactly which IDs were evaluated.

In [ ]:
final_rows = []
final_predictions, final_targets = None, None
for policy in policies:
    model.transform.min_size = (policy['min_size'],)
    model.transform.max_size = policy['max_size']
    predictions, targets = predict_loader(model, loader(parts['evaluation'], best_cfg), DEVICE)
    metrics = evaluate_detections(predictions, targets)
    write_json(RUN_DIR / f"final_{policy['name']}_metrics.json", metrics)
    final_rows.append({**policy, 'map50': metrics['map50'],
                       'map50_95_voc': metrics['map50_95_voc'],
                       'images': metrics['images'], 'classes_with_gt': metrics['classes_with_gt']})
    if policy['name'] == SELECTED_POLICY['name']:
        final_predictions, final_targets = predictions, targets
final_table = pd.DataFrame(final_rows)
final_table.to_csv(RUN_DIR / 'final_comparison.csv', index=False)
display(final_table)
final_errors = pd.DataFrame(error_analysis(final_predictions, final_targets))
final_errors.to_csv(RUN_DIR / 'final_errors.csv', index=False)
for i in range(min(3, len(parts['evaluation']))):
    record = parts['evaluation'][i]
    with Image.open(record['path']) as im:
        annotated = draw_detections(im, final_predictions[i])
        annotated.save(RUN_DIR / f"prediction_{record['id']}.jpg")
        display(annotated)

## 9. Export and use on new images
Export a portable state-dict checkpoint plus a deployment policy for the local Streamlit app. Reloading does not fetch weights. It uses the same class mapping, preprocessing and selected resolution. Copy/download the entire `artifacts/` folder before a free GPU session ends. The app defaults to CPU inference and runs locally without a paid service.

The best checkpoint is selected using the tuning split and is not retrained on tuning data; this keeps the selected checkpoint directly comparable to the measured metrics.

In [ ]:
Path('artifacts').mkdir(exist_ok=True)
shutil.copy2(BEST_CHECKPOINT, 'artifacts/best.pt')
write_json('artifacts/deployment.json', {
    **SELECTED_POLICY, 'source_run': str(RUN_DIR),
    'quick_run': QUICK_RUN, 'classes': list(CLASSES),
    'checkpoint': 'best.pt', 'quality_budget_absolute_map50_drop': MAX_MAP50_DROP})
# Check exported artifacts work without pretrained downloads.
deployed_model, _, _ = load_checkpoint('artifacts/best.pt', DEVICE)
deployed_model.transform.min_size = (SELECTED_POLICY['min_size'],)
deployed_model.transform.max_size = SELECTED_POLICY['max_size']
print('Exported artifacts/best.pt and artifacts/deployment.json')
print('From a terminal in this folder: streamlit run app.py')

In [ ]:
# Optional: run on your own image, using precisely the app's preprocessing.
NEW_IMAGE_PATH = None  # e.g. 'my_photo.jpg'
if NEW_IMAGE_PATH:
    from PIL import ImageOps
    from torchvision.transforms.functional import pil_to_tensor
    image = ImageOps.exif_transpose(Image.open(NEW_IMAGE_PATH)).convert('RGB')
    with torch.inference_mode():
        prediction = deployed_model([pil_to_tensor(image).float().to(DEVICE)/255.])[0]
    display(draw_detections(image, prediction, .5))

## 10. Object results and report

The next cell writes the object-task results from your run. Complete `reports/TECHNICAL_REPORT.md` with real observations and `AI_USAGE.md` with the assistance/verification you actually used.

**Continue below for layout training.** The notebook preserves the object results and checkpoint before switching the working variables to the body-part task. Both deployment models are assembled at the end.


In [ ]:
summary = {
    'run_directory': str(RUN_DIR), 'quick_run': QUICK_RUN,
    'split_sizes': {k: len(v) for k,v in parts.items()},
    'pretrained_baseline_tune_map50': baseline_metrics['map50'],
    'winner': str(best['run']), 'winner_tune_map50': float(best['best_tune_map50']),
    'selected_policy': SELECTED_POLICY,
    'final_comparison': final_table.to_dict(orient='records'),
    'status': 'Smoke run only; run full data before submission.' if QUICK_RUN else 'Measured full-split run; review technical report.'
}
write_json(RUN_DIR / 'results_summary.json', summary)
print(json.dumps(summary, indent=2))
# Optional Colab download (includes model, metrics, audit and split manifests):
# shutil.make_archive('trained_artifacts', 'zip', 'artifacts')
# from google.colab import files
# files.download('trained_artifacts.zip')

In [ ]:
# Preserve the object task's outputs before starting the layout task.
object_final_table = final_table.copy()
object_summary = summary.copy()
object_run_dir = RUN_DIR
object_checkpoint = BEST_CHECKPOINT
object_policy = SELECTED_POLICY.copy()
# Release object model instances from memory; their checkpoints are already exported.
del model, deployed_model
gc.collect()
if DEVICE.type == 'cuda':
    torch.cuda.empty_cache()
print('Object results preserved:', object_run_dir)

## 11. Person-layout setup

The following sections are part of this **same notebook**. The dataset was downloaded above and is reused. `LAYOUT_CFG` and the shared `QUICK_RUN` flag were configured in section 0.

The layout model gets its own run directory. It does not overwrite the object checkpoint. New head/hand/foot rows are initialized randomly and require training. Read the coverage exclusions below before interpreting results.

In [ ]:
CFG = LAYOUT_CFG
RUN_DIR = object_run_dir.parent / f'layout_{RUN_NAME}'
RUN_DIR.mkdir(parents=True, exist_ok=False)
seed_everything(CFG.seed)
write_json(RUN_DIR / 'environment.json', {
    'config': CFG.dictionary(), 'quick_run': QUICK_RUN,
    'torch': str(torch.__version__), 'torchvision': str(torchvision.__version__),
    'device': str(DEVICE), 'dataset': str(VOC_DIR),
})
print('Layout device:', DEVICE, '| Quick run:', QUICK_RUN, '| Output:', RUN_DIR)

## 12. Data cleaning — use the Layout split and nested part labels
`ImageSets/Layout/{train,val}.txt` contains `(image_id, object_index)` pairs. The object index is **1-based across all top-level XML objects**. We read only listed persons and their nested `<part>` labels, not ordinary `Main` labels.

Layout annotation is incomplete across people. Training/evaluation are therefore person-conditioned; unlisted people are not labeled as target persons. Multiple people from the same image stay together when splitting. Neighboring people can still appear in crops and confuse the model.

Convert 1-based inclusive boxes to half-open XYXY; validate finite coordinates, classes/counts and image bounds; deduplicate decoded images while retaining all persons in kept images. Layout val is preserved first. Log every rejection/repair.

Crops use person boxes padded by 35% on each side, independent of part labels. Parts may legitimately extend beyond the person box. If any annotated part extends outside the fixed crop, reject and log the whole sample rather than clipping away its label. This creates a coverage limitation: disclose exclusions and evaluated person IDs. Adjust padding before experiments if needed, then regenerate all splits. No near-duplicate or COCO image-overlap audit is performed.

In [ ]:
# Reuse VOC_DIR from the object data-cleaning section; no second download.
clean, events = audit_layout(VOC_DIR, RUN_DIR, padding=CFG.crop_padding)
print('Kept persons:', {k:len(v) for k,v in clean.items()})
print('Audit events:', len(events))
display(pd.DataFrame(events).head(20))

## 13. Preprocessing and image-grouped splits
Fit/tune are an 80/20 split of Layout train **images**. Layout val is final evaluation. Crops from one image never cross splits. Quick limits apply afterward and count persons, not source images.

RGB pixels become float tensors in `[0,1]`; the detector normalizes/resizes internally. Horizontal flip updates all part boxes. One head and up to two hands/two feet are supervised. No left/right labels are invented.

In [ ]:
parts = make_layout_splits(clean, CFG, RUN_DIR)
counts = pd.DataFrame({k:class_counts(v, LAYOUT_CLASSES) for k,v in parts.items()})
display(counts)
print('Persons:', {k:len(v) for k,v in parts.items()})
if (counts == 0).any().any():
    print('WARNING: class coverage is incomplete; use full mode for reporting.')
counts.plot.bar(title='Part counts', figsize=(8,3)); plt.tight_layout(); plt.show()
image_tensor, target = VOCDataset(parts['fit'])[0]
sample_image = Image.fromarray((image_tensor.permute(1,2,0).numpy()*255).astype('uint8'))
display(draw_detections(sample_image, {**target, 'scores':torch.ones(len(target['boxes']))},
                        classes=LAYOUT_CLASSES))

## 14. Fine-tuning
Train the new part head and RPN/ROI components with the backbone/FPN frozen as the first experiment. SGD, schedule, gradient clipping, CUDA AMP and early stopping reuse the object-training implementation. Select the best epoch by **conditional tune mAP50 over the three part classes**.

The head starts with random head/hand/foot rows; the old 20-class checkpoint cannot be relabeled into a part detector. Full training has a default budget of 12 epochs per trial. This budget is a starting point, not evidence of convergence; inspect real curves. Quick mode cannot establish usable part detection quality.

In [ ]:
frozen = replace(
    CFG, lr=.005, freeze_backbone=False, flip_probability=.5
)
results = [train_experiment(frozen, parts['fit'], parts['tune'], RUN_DIR/'frozen', DEVICE)]

## 15. Hyperparameter tuning and ablation
Compare unfrozen-backbone learning rates and a no-flip ablation on identical fit/tune records, seeds and epoch budgets. Frozen versus full at LR .005 isolates backbone training; the two full runs isolate LR; full_no_flip versus full_lr005 isolates augmentation. Select only from tune scores. Repeat seeds for stronger evidence.

In [ ]:
trials = {
    'full_lr005':replace(CFG, lr=.005, freeze_backbone=False),
    'full_lr001':replace(CFG, lr=.001, freeze_backbone=False),
    'full_no_flip':replace(CFG, lr=.005, freeze_backbone=False, flip_probability=0.),
}
for name, cfg in trials.items():
    results.append(train_experiment(cfg, parts['fit'], parts['tune'], RUN_DIR/name, DEVICE))
experiments = pd.DataFrame(results).sort_values('best_tune_map50', ascending=False)
experiments.to_csv(RUN_DIR/'layout_experiments.csv', index=False)
display(experiments[['run','lr','freeze_backbone','flip_probability','best_tune_map50','seconds']])
BEST_CHECKPOINT = Path(experiments.iloc[0]['checkpoint'])
model, best_cfg, metadata = load_checkpoint(BEST_CHECKPOINT, DEVICE)
for result in results:
    history = pd.read_json(RUN_DIR/result['run']/'history.json')
    plt.plot(history['epoch'], history['tune_map50'], marker='o', label=result['run'])
plt.xlabel('Epoch'); plt.ylabel('Conditional tuning mAP50'); plt.legend(); plt.show()

## 16. Metrics and error analysis
Compute AP50 for head/hand/foot and their mean; `map50_95_voc` is the custom VOC-style IoU average, not official COCO AP. Absent classes are undefined/excluded. Each ground-truth person crop is a separate evaluation sample, so boxes are matched only to that person's labels. There are no difficult layout objects.

Predictions are capped to one head/two hands/two feet in both evaluation and serving, retaining low-score predictions for AP ranking. The UI subsequently applies a confidence threshold. This is **conditional person-crop AP**, not official VOC layout AP: the official protocol uses layout-level confidence and full-image boxes; our excluded samples and crop context also differ. It does not measure errors from the first-stage person detector.

Inspect missed parts, localization mistakes and confusion with neighboring people's parts. A part below threshold is 'not detected', not proof that the part is invisible.

In [ ]:
pred, gt = predict_loader(model, loader(parts['tune'], best_cfg), DEVICE)
quality = evaluate_detections(pred, gt, classes=LAYOUT_CLASSES)
write_json(RUN_DIR/'layout_tune_metrics.json', quality)
display(pd.Series(quality['per_class_ap50']).to_frame('AP50'))
errors = pd.DataFrame(error_analysis(pred, gt))
errors.to_csv(RUN_DIR/'layout_tune_errors.csv', index=False)
display(errors.drop(columns='image_index').sum().to_frame('count'))
for i in range(min(3,len(parts['tune']))):
    tensor, _ = VOCDataset(parts['tune'])[i]
    image = Image.fromarray((tensor.permute(1,2,0).numpy()*255).astype('uint8'))
    display(draw_detections(image, pred[i], classes=LAYOUT_CLASSES))

## 17. Inference optimization
Compare original and smaller part-model resolution on tune crops. Choose the fastest median latency within a predeclared 0.02 absolute conditional tune mAP50 loss. Timing uses the part model only, with warmed, synchronized batch-one inference. It excludes the first-stage person detector, image decoding and UI. Keep original if the alternative fails quality or is slower. Checkpoint size is unchanged.

In [ ]:
policies = [
    {'name':'original', 'min_size':best_cfg.min_size, 'max_size':best_cfg.max_size},
    {'name':'smaller', 'min_size':max(160,best_cfg.min_size*2//3),
     'max_size':max(320,best_cfg.max_size*2//3)},
]
rows=[]
for policy in policies:
    model.transform.min_size=(policy['min_size'],)
    model.transform.max_size=policy['max_size']
    p,t = predict_loader(model, loader(parts['tune'],best_cfg), DEVICE)
    q = evaluate_detections(p,t,classes=LAYOUT_CLASSES)
    timing = benchmark(model, VOCDataset(parts['tune']), DEVICE, repeats=10 if QUICK_RUN else 30)
    rows.append({**policy, 'tune_map50':q['map50'], **timing})
optimization = pd.DataFrame(rows)
assert rows[0]['tune_map50'] is not None
eligible = optimization[optimization.tune_map50 >= rows[0]['tune_map50']-.02]
selected_name = eligible.sort_values('median_ms',kind='stable').iloc[0]['name']
SELECTED_POLICY = next(p for p in policies if p['name']==selected_name)
optimization.to_csv(RUN_DIR/'layout_optimization.csv',index=False)
display(optimization)
print('Selected:', SELECTED_POLICY)

## 18. Final conditional evaluation
Model and deployment resolution are now fixed. Report both resolutions on untouched Layout val persons; do not use these results to retune the selection. Full mode evaluates all persons retained by the documented audit. Quick mode evaluates only a subset.

In [ ]:
final_rows=[]
for policy in policies:
    model.transform.min_size=(policy['min_size'],)
    model.transform.max_size=policy['max_size']
    p,t = predict_loader(model, loader(parts['evaluation'], best_cfg), DEVICE)
    q = evaluate_detections(p,t,classes=LAYOUT_CLASSES)
    write_json(RUN_DIR/f"layout_final_{policy['name']}.json",q)
    final_rows.append({**policy, 'conditional_map50':q['map50'],
        'conditional_map50_95_voc':q['map50_95_voc'], 'persons':q['images'],
        'classes_with_gt':q['classes_with_gt'], **q['per_class_ap50']})
final_table=pd.DataFrame(final_rows)
final_table.to_csv(RUN_DIR/'layout_final_comparison.csv',index=False)
display(final_table)
print('Quick run:', QUICK_RUN)

## 19. Export and demo
Export `layout_best.pt` and `layout_deployment.json`. The demo requires **four files** in `artifacts/`: your original object model's `best.pt` and `deployment.json`, plus these two layout files. These sections do not overwrite the object model exported earlier.

A real image follows: image → object/person detector → padded person crops → part detector → image-coordinate boxes grouped by person ID. Crowded scenes can cause part ownership mistakes; missed people cannot receive part predictions. Invisible parts are not forced into the result.

The Streamlit UI shows a task selector, upload, Detect button and annotated result; details/settings are collapsed. A real FastAPI service also provides `POST /predict` (JSON) and `POST /predict/image` (annotated PNG). Run these locally after downloading artifacts; no paid hosting or Colab web UI tunnel is needed.

In [ ]:
Path('artifacts').mkdir(exist_ok=True)
shutil.copy2(BEST_CHECKPOINT,'artifacts/layout_best.pt')
write_json('artifacts/layout_deployment.json',{
    **SELECTED_POLICY, 'task':'layout','classes':list(LAYOUT_CLASSES),
    'quick_run':QUICK_RUN,'crop_padding':best_cfg.crop_padding,'source_run':str(RUN_DIR),
    'metric_scope':'conditional AP given ground-truth person crops; not end-to-end AP'})
# Verify strict reload without pretrained downloads.
loaded,_,_ = load_checkpoint('artifacts/layout_best.pt',DEVICE)
del loaded
print('Layout exported. Object best.pt + deployment.json were exported in section 9.')
print('Local UI: python -m streamlit run app.py')
print('Local API: python -m uvicorn api:app --host 127.0.0.1 --port 8000')
# Colab download:
# from google.colab import files
# files.download('artifacts/layout_best.pt')
# files.download('artifacts/layout_deployment.json')

### Layout report and limits
Complete `reports/LAYOUT_REPORT.md` with measured counts, exclusions, split IDs, hyperparameters, per-part AP, errors and speed/quality results. The original 74.91% quick-run object mAP does not describe this new task. No part AP is known until you train and evaluate.

Strong points: explicit person ownership in labels, image-grouped splits, pretrained feature reuse, editable config, shared inference for UI/API. Limits: small dataset, cropped/conditional rather than official evaluation, no end-to-end quality estimate, partial annotations and crop exclusions, neighboring-person confusion, unverified pretraining-image overlap, no exact interrupted optimizer resume.

[Official VOC person-layout specification](https://www.robots.ox.ac.uk/~vgg/projects/pascal/VOC/voc2012/htmldoc/index.html) · [Torchvision detector](https://docs.pytorch.org/vision/0.23/models/generated/torchvision.models.detection.fasterrcnn_mobilenet_v3_large_fpn.html)

## 20. Combined results and deployment export

Object metrics use full images; layout metrics use known person crops. They measure different problems and must be reported separately. The table below identifies the sample unit and evaluation scope. Quick-mode scores are only smoke-run evidence.

The notebook has now trained and exported both models. Download `trained_models.zip`, extract it into the local project folder, then run either the plain Streamlit demo or the REST API. The archive contains the four deployment files; full trial logs and cleaning/split reports remain in their run directories.

In [ ]:
layout_final_table = final_table.copy()
layout_run_dir = RUN_DIR
object_results = object_final_table.copy()
object_results.insert(0, 'task', 'objects')
object_results.insert(1, 'evaluation_scope', 'held-out full images')
layout_results = layout_final_table.rename(columns={
    'conditional_map50': 'map50',
    'conditional_map50_95_voc': 'map50_95_voc',
}).copy()
layout_results.insert(0, 'task', 'person layout')
layout_results.insert(1, 'evaluation_scope', 'known-person crops; not end-to-end')
combined_results = pd.concat([object_results, layout_results], ignore_index=True)
display(combined_results)
write_json(Path('artifacts') / 'combined_results.json', {
    'quick_run': QUICK_RUN,
    'object_run': str(object_run_dir), 'layout_run': str(layout_run_dir),
    'object_results': object_final_table.to_dict(orient='records'),
    'layout_results': layout_final_table.to_dict(orient='records'),
})
import zipfile
deployment_files = ['best.pt', 'deployment.json', 'layout_best.pt', 'layout_deployment.json']
for filename in deployment_files:
    assert (Path('artifacts') / filename).is_file(), f'Missing export: {filename}'
with zipfile.ZipFile('trained_models.zip', 'w', zipfile.ZIP_DEFLATED) as archive:
    for filename in deployment_files:
        archive.write(Path('artifacts') / filename, arcname=f'artifacts/{filename}')
print('Quick run:', QUICK_RUN)
print('Both tasks exported to trained_models.zip.')
print('Local visual demo: python -m streamlit run app.py')
print('Local REST API: python -m uvicorn api:app --host 127.0.0.1 --port 8000')
# On Colab, uncomment to download the two trained deployment models and policies:
# from google.colab import files
# files.download('trained_models.zip')

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

for folder_name in ["voc_detection_runs", "voc_layout_runs"]:
    folder = Path("/content/drive/MyDrive") / folder_name
    if folder.exists():
        for path in folder.rglob("*.pt"):
            print(path)


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import shutil

shutil.copy2(
    "trained_models.zip",
    "/content/drive/MyDrive/trained_models.zip",
)

## References
- [Torchvision Faster R-CNN MobileNetV3 documentation](https://docs.pytorch.org/vision/0.23/models/generated/torchvision.models.detection.fasterrcnn_mobilenet_v3_large_fpn.html)
- [Torchvision detection fine-tuning tutorial](https://docs.pytorch.org/tutorials/intermediate/torchvision_tutorial.html)
- [VOC 2012 development kit and evaluation rules](https://www.robots.ox.ac.uk/~vgg/projects/pascal/VOC/voc2012/htmldoc/index.html)
- [Matching PyTorch/Torchvision installation pairs](https://pytorch.org/get-started/previous-versions/)